# 7.5 Text Mining in Higher Ed – Sentiment Analysis


## Introduction


In our previous modules, we developed the technical ability to clean student text and group it into thematic clusters using Topic Modeling. However, knowing *what* a student is talking about is only half the story. To truly understand the student experience, we must also evaluate the **emotional tone** of their feedback.

**Sentiment Analysis** is the process of computationally identifying and categorizing opinions expressed in text to determine whether the writer's attitude is positive, negative, or neutral. In Higher Education, this allows us to move beyond simple satisfaction scores and quantify the "pulse" of the student body across thousands of open-ended comments.

**In this module, we will explore:**
* The strategic **strengths and limitations** of sentiment analysis in an institutional context.
* **Lexicon-Based Sentiment (VADER):** A rule-based approach that uses a pre-defined dictionary of "weighted" words to score sentiment without needing training data.
* **Supervised Sentiment Classification:** Using your existing labeled data to train a custom **Logistic Regression** model paired with **TF-IDF** vectors.
* **Interpretation & Communication:** How to pair sentiment scores with topics to provide a multidimensional report for campus stakeholders.

<br>

#### **Guiding Questions**
1. How can we estimate whether student comments trend positive, negative, or mixed—at scale?
2. When is sentiment useful for institutional research, and when can it be misleading?

<br>

#### **Learning Objectives**
By the end of this notebook, you will be able to:
* **Distinguish** between lexicon-based and supervised machine learning approaches to sentiment.
* **Implement VADER** to quickly generate sentiment polarity scores.
* **Build a classification pipeline** to predict sentiment labels using scikit-learn.
* **Visualize** sentiment distributions to identify institutional "pain points" or successes.

> **Instructor Perspective:** Sentiment analysis is a powerful "signal," but it is not a replacement for human empathy. Your role as an analyst is to use these scores to find the "where" and "what," then dive into the qualitative data to understand the "why."

<br>



## 1. What Sentiment Analysis Can (and Can’t) Do in IR



**Sentiment analysis** estimates the emotional tone of text (often simplified as *positive*, *negative*, or *neutral*).

Common IR use cases:
- Monitoring shifts in student satisfaction across terms
- Summarizing course evaluation comments at scale
- Adding a *“tone”* feature to a larger student success model

Important caveats:
- Sentiment tools can misread sarcasm, cultural language differences, or domain-specific terms.
- “Negative” comments are not always *bad*—they may point to real barriers that need attention.
- Always pair sentiment with **themes/topics** and **examples**.


## 2. Setup and Data Preparation



We’ll import our survey data that includes ordinal responses, as well as unstructured responses with clear positive/negative signals.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import random
import numpy as np
import pandas as pd
import plotly.express as px

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

pd.options.display.max_columns = None


To demonstrate supervised machine learning later in this module, we will generate a set of synthetic comments where the emotional tone is tied to academic markers.

**In a professional Institutional Research deployment, these "gold standard" labels are typically derived from:**
* **Manual Human Coding:** Researchers reviewing a subset of comments using a standardized rubric.
* **Outcome Proxies:** Using high/low satisfaction ratings from structured survey questions as a label for the accompanying text.
* **Historical Data:** Utilizing previously categorized longitudinal datasets.

In [11]:
filepath = '/content/drive/MyDrive/IR ML Cert/MLCert Course 3/Course 3 Data/'
ML_Survey_Data = pd.read_csv(f'{filepath}ML_Survey_Data.csv')
display(ML_Survey_Data)

,SID,NSSE_Q1,NSSE_Q2,NSSE_Q3,NSSE_Q4,NSSE_Q5,NSSE_Q6,NSSE_Q7,NSSE_Q8,NSSE_Q9,NSSE_Q10,Free_Response_Text
0,UHDOP5522,3,3,3,3,2,2,3,2,3,3,"I feel prepared in some areas, but there are topics where I need more practice or review."
1,UHE842CU6,1,2,1,2,2,1,2,2,2,2,"I often start assignments too late, and then I’m rushing to finish instead of learning the material."
2,UHJFT1JAB,3,2,2,2,2,3,3,2,2,3,"When multiple deadlines hit at once, I have to prioritize carefully to keep up."
3,UHKF05TAF,3,3,3,3,2,3,3,2,3,3,"Group work can be helpful, but it depends on how organized the team is."
4,UHKKQ8UY5,3,3,3,2,2,3,2,2,3,2,"Some classes feel more challenging than I expected, and I’m learning how to meet those expectations."
...,...,...,...,...,...,...,...,...,...,...,...,...
19839,N2HOWBXJM,2,2,2,2,2,2,2,2,3,2,"The workload can feel heavy at times, but I can manage when I start assignments early."
19840,N2JGRD0V6,4,3,3,3,3,3,3,3,3,3,"Some classes feel more challenging than I expected, and I’m learning how to meet those expectations."
19841,N2K6479P0,4,3,3,3,3,3,3,3,3,3,"College is an interesting experience, and I'm discovering what I'm passionate about."
19842,N2LXOSYTW,1,1,1,1,1,1,1,1,1,1,"Sometimes I don’t understand what instructors expect, and I wish I had more guidance earlier in the course."


## 3. Lexicon-Based Sentiment (VADER)


**VADER (Valence Aware Dictionary and sEntiment Reasoner)** is a rule-based sentiment analysis tool that measures the emotional polarity of text. At a high level, it works by scanning text for words in a built-in sentiment dictionary and then applying linguistic rules to adjust for tone and emphasis.

Originally developed for social media, VADER is exceptionally effective for the brief, informal comments commonly found in course evaluations, advising notes, and open-ended survey responses. As a **lexicon-based** approach, it relies on a predefined dictionary of words that have been manually scored for their emotional intensity.

<br>

#### **How VADER Assigns Value**
VADER uses a built-in lexicon where words are assigned specific sentiment scores:

| Word | Score |
| :--- | :--- |
| **excellent** | +3.0 |
| **good** | +1.9 |
| **terrible** | -2.5 |
| **disappointing** | -2.0 |

* It accounts for **intensifiers** (e.g., "extremely good") and **negations** (e.g., "not good") to refine the final output.

<br>

#### **Understanding VADER Scores**
When VADER analyzes text, it returns four scores: **positive, negative, neutral, and a compound score**.

The **Compound Score** ranges from **-1.0 (Most Negative)** to **+1.0 (Most Positive)** and represents the overall tone of the text. For example, a comment like *“I enjoyed the class, but the workload was overwhelming”* might receive a slightly positive compound score if the overall tone leans positive, even though it contains some negative language.

**Standard Interpretation Thresholds:**
* **Positive:** Score $\ge 0.05$
* **Neutral:** Score between $-0.05$ and $0.05$
* **Negative:** Score $\le -0.05$

<br>

| Pros | Cons |
| :--- | :--- |
| No training data or manual labels required. | May miss domain-specific nuance (e.g., "challenging"). |
| Extremely fast and scalable for large datasets. | Can struggle with heavy sarcasm or complex metaphors. |
| Transparent and easy to explain to stakeholders. | Generic lexicon is not tailored specifically to academic life. |

> **Instructor Perspective:** VADER is your "baseline." It gives you an immediate pulse of the student voice without any manual coding. Use the Compound Score to identify broad trends before deciding if a more complex, custom-trained model is necessary.

---


In [ ]:
import nltk
nltk.download('vader_lexicon')
from nltk.sentiment import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()

df_Sentiment = ML_Survey_Data[['SID','Free_Response_Text']]

df_Sentiment['vader_compound'] = df_Sentiment['Free_Response_Text'].apply(lambda t: sia.polarity_scores(t)['compound'])

# Convert compound score to a simple label
def vader_label(score, pos_thresh=0.05, neg_thresh=-0.05):
    if score >= pos_thresh:
        return 'positive'
    if score <= neg_thresh:
        return 'negative'
    return 'neutral'

df_Sentiment['vader_label'] = df_Sentiment['vader_compound'].apply(vader_label)
display(df_Sentiment[['Free_Response_Text','vader_compound','vader_label']].head(10))


## 4. Interpreting and Communicating Results


In Institutional Research (IR), sentiment scores are most effective when they are paired with other data points to provide a complete story. A single "negative" score tells us a student is unhappy, but it doesn't tell us why.

**To create an effective institutional report, use this pattern:**
1. **Trend Analysis:** Track sentiment by term, program, or specific course to identify shifts over time.
2. **Thematic Pairing:** Combine sentiment with the **topics** discovered in the previous module (7.4) to understand the specific drivers of student emotion.
3. **Qualitative Evidence:** Include a small number of de-identified, representative comments to give "voice" to the quantitative charts.

<br>

> **Key Insight:** Below, we visualize the sentiment distribution for this cohort. This high-level view allows stakeholders to quickly see the "ratio" of student sentiment, which can then be used to prioritize areas for deeper qualitative review.

In [13]:
sent_counts = df_Sentiment['vader_label'].value_counts().reset_index()
sent_counts.columns = ['vader_label','Count']
fig = px.bar(sent_counts, x='vader_label', y='Count', title='Sentiment Distribution (VADER)')
fig.show()


## 5. Custom Sentiment & Validation


While VADER is an excellent baseline, it cannot "learn" your specific campus context. In this section, we generate a "Ground Truth" dataset where sentiment labels are tied directly to student academic markers (GPA).

This allows us to perform a **Validation Analysis** to see how well the generic VADER model performs compared to a custom-defined sentiment logic.

In [15]:
# Synthetic sentiment-linked comments
df_training = pd.read_csv(f'{filepath}training.csv')

sent_df = df_training[['SID']].copy()

positive_bank = [
    'I really enjoyed this course and felt supported by my instructors.',
    'The assignments were challenging in a good way, and I learned a lot.',
    'Advising was helpful and I feel confident about my academic plan.',
    'I feel connected on campus and my classes have been engaging.'
]
neutral_bank = [
    'The semester was okay overall, with some ups and downs.',
    'Some parts of the course were useful, and others were less clear.',
    'I managed my workload, but it was sometimes difficult to stay organized.',
    'My experience was mixed depending on the class and the week.'
]
negative_bank = [
    'I struggled a lot and often felt like I did not know where to get help.',
    'The workload was overwhelming and I felt stressed most weeks.',
    'I had a hard time understanding expectations and felt unsupported.',
    'I felt disconnected and frustrated with how things were communicated.'
]

def make_sentiment_comment(gpa_norm):
    # More likely positive when GPA_norm is higher; more likely negative when lower
    if gpa_norm > 3.7:
        label = 'positive'
        text = random.choice(positive_bank)
    elif gpa_norm < 3.4:
        label = 'negative'
        text = random.choice(negative_bank)
    else:
        # mixed middle group
        label = random.choice(['neutral','positive','negative'])
        text = random.choice(neutral_bank if label=='neutral' else (positive_bank if label=='positive' else negative_bank))
    return text, label

rows=[]
for sid, g in zip(df_training['SID'], df_training['HS_GPA']):
    text, label = make_sentiment_comment(float(g))
    rows.append((sid, text, label))

sent_df = pd.DataFrame(rows, columns=['SID','Comment','Labeled Sentiment'])

sent_df1 = pd.merge(sent_df,df_Sentiment[['SID', 'Free_Response_Text', 'vader_label']],on="SID",how="left")
display(sent_df1.head())
sent_df1['Labeled Sentiment'].value_counts()

,SID,Comment,Labeled Sentiment,Free_Response_Text,vader_label
0,UHDOP5522,"The assignments were challenging in a good way, and I learned a lot.",positive,"I feel prepared in some areas, but there are topics where I need more practice or review.",positive
1,UHE842CU6,I felt disconnected and frustrated with how things were communicated.,negative,"I often start assignments too late, and then I’m rushing to finish instead of learning the material.",neutral
2,UHJFT1JAB,The workload was overwhelming and I felt stressed most weeks.,negative,"When multiple deadlines hit at once, I have to prioritize carefully to keep up.",positive
3,UHKF05TAF,"I managed my workload, but it was sometimes difficult to stay organized.",neutral,"Group work can be helpful, but it depends on how organized the team is.",positive
4,UHKKQ8UY5,Advising was helpful and I feel confident about my academic plan.,positive,"Some classes feel more challenging than I expected, and I’m learning how to meet those expectations.",positive


,count
Labeled Sentiment,
positive,11289
negative,6883
neutral,1712


We can already see that the labeled sentiments are not as optimistic as our VADER labels. Let's proceed to compare across the training set.


## 6. Model Comparison (Crosstab Analysis)


The final step in any sentiment project is validation. By using a **Crosstab**, we can see exactly where VADER aligns with our ground-truth labels and where it struggles. This tells the IR analyst whether they can trust the "off-the-shelf" tool or if they need to build a custom supervised model.

In [16]:
ct = pd.crosstab(sent_df1['Labeled Sentiment'], sent_df1['vader_label'])
ct

vader_label,negative,neutral,positive
Labeled Sentiment,,,
negative,999,1314,4570
neutral,268,280,1164
positive,1502,1362,8425


Similar to a confusion matrix, we hope to see the largest numerical values on the diagonal, and smaller values off diagonal. this is the case for the majority class, "positive", but the VADER model still struggles here and in the other two classes as well. This mismatch confirms our suspicions. Let's take a look at the head and tail observations to see if we can pinpoint the issue here.

In [17]:
pd.set_option('display.max_colwidth', None)
sent_df1

,SID,Comment,Labeled Sentiment,Free_Response_Text,vader_label
0,UHDOP5522,"The assignments were challenging in a good way, and I learned a lot.",positive,"I feel prepared in some areas, but there are topics where I need more practice or review.",positive
1,UHE842CU6,I felt disconnected and frustrated with how things were communicated.,negative,"I often start assignments too late, and then I’m rushing to finish instead of learning the material.",neutral
2,UHJFT1JAB,The workload was overwhelming and I felt stressed most weeks.,negative,"When multiple deadlines hit at once, I have to prioritize carefully to keep up.",positive
3,UHKF05TAF,"I managed my workload, but it was sometimes difficult to stay organized.",neutral,"Group work can be helpful, but it depends on how organized the team is.",positive
4,UHKKQ8UY5,Advising was helpful and I feel confident about my academic plan.,positive,"Some classes feel more challenging than I expected, and I’m learning how to meet those expectations.",positive
...,...,...,...,...,...
19879,N2HOWBXJM,I felt disconnected and frustrated with how things were communicated.,negative,"The workload can feel heavy at times, but I can manage when I start assignments early.",neutral
19880,N2JGRD0V6,"The assignments were challenging in a good way, and I learned a lot.",positive,"Some classes feel more challenging than I expected, and I’m learning how to meet those expectations.",positive
19881,N2K6479P0,I really enjoyed this course and felt supported by my instructors.,positive,"College is an interesting experience, and I'm discovering what I'm passionate about.",positive
19882,N2LXOSYTW,I had a hard time understanding expectations and felt unsupported.,negative,"Sometimes I don’t understand what instructors expect, and I wish I had more guidance earlier in the course.",positive


Drilling down, we can clearly see that the VADER model, which is based on a generic lexicon, may be failing to catch the nuances of word meaning when applied to higher ed. However, there could be the temptation to report this overly optimistic presentation of student opinion as truth, as opposed to the mathematical model that it actually is. As always, ethics is central from start to finish in the machine learning cycle.

As mentioned above, another way to validate the VADER results is to use existing survey results. In the assignment, you'll be directed on how to use the **ordinal responses** to do this, and compare results with the labeled sentiment approach.


## 7. Wrap-Up


In this module, we explored how to evaluate the emotional "pulse" of student feedback using two distinct pathways:
* **Lexicon-based sentiment (VADER):** A fast, rule-based approach that requires no training data and provides an immediate overview of student sentiment.
* **Supervised sentiment classification:** A custom approach (using TF-IDF and Logistic Regression) that can be trained to recognize the specific linguistic nuances of your institution.

<br>

 **Key Takeaways for the IR Professional:**
* **Sentiment as a Signal:** Treat sentiment scores as a "flag" for further investigation rather than a definitive judgment.
* **Context is King:** Always pair sentiment results with the **Topic Models** from the previous module (7.4) to understand the "why" behind the emotion.
* **Validation:** The final code blocks below demonstrate how to compare model results against "ground truth" labels using a crosstab—a vital step for ensuring your reporting is accurate.
* **Ethics:** Models can be construed to deliver almost any message. An ethical approach demands wholistic, integral handling of data reporting.

> **Key Concept:** You have now mastered the three pillars of text mining: **Vectorization**, **Topic Modeling**, and **Sentiment Analysis**. By combining these techniques, you can transform thousands of unstructured comments into a clear, multi-dimensional story of the student experience.